In [1]:
import pandas as pd

In [2]:
raw_df=pd.read_csv('insurance.csv')

In [3]:
raw_df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
inputs=raw_df[['age','sex','bmi','children','smoker','region']]
target=raw_df['charges']

In [5]:
numerical_cols=['age','bmi','children']

In [6]:
from sklearn.model_selection import train_test_split
inputs_train,input_test,target_train,target_test=train_test_split(inputs, target, test_size=0.25, random_state=42)

In [7]:
dict1={'no':0,'yes':1}

In [8]:
dict2={'male':1,'female':0}

In [9]:
dict3={'southeast':1,'southwest':2,'northeast':3,'northwest':4}

In [10]:
inputs_train['smoker']=inputs_train.smoker.map(dict1)
input_test['smoker']=input_test.smoker.map(dict1)

In [11]:
inputs_train['sex']=inputs_train.sex.map(dict2)
input_test['sex']=input_test.sex.map(dict2)

In [12]:
inputs_train['region']=inputs_train.region.map(dict3)
input_test['region']=input_test.region.map(dict3)

In [13]:
input_test

,age,sex,bmi,children,smoker,region
764,45,0,25.175,2,0,3
887,36,0,30.020,0,0,4
890,64,0,26.885,0,1,4
1293,46,1,25.745,3,0,4
259,19,1,31.920,0,1,4
...,...,...,...,...,...,...
342,60,0,27.550,0,0,3
308,58,1,34.865,0,0,3
1128,34,1,32.800,1,0,2
503,19,1,30.250,0,1,1


In [14]:
inputs_train

,age,sex,bmi,children,smoker,region
693,24,1,23.655,0,0,4
1297,28,0,26.510,2,0,1
634,51,1,39.700,1,0,2
1022,47,1,36.080,1,1,1
178,46,0,28.900,2,0,2
...,...,...,...,...,...,...
1095,18,0,31.350,4,0,3
1130,39,0,23.870,5,0,1
1294,58,1,25.175,0,0,3
860,37,0,47.600,2,1,2


In [15]:
from sklearn.ensemble import RandomForestRegressor

In [46]:
model=RandomForestRegressor(random_state=42)

In [47]:
model.fit(inputs_train,target_train)

RandomForestRegressor(random_state=42)

In [48]:
preds=model.predict(inputs_train)

In [49]:
preds

array([ 3507.1931982,  4268.1642145,  9570.6489402, ..., 12013.888135 ,
       44955.6079538, 10844.4376388])

In [50]:
from sklearn.metrics import r2_score

In [51]:
lossy=r2_score(target_train,preds)

In [52]:
lossy

0.9757726664985225

In [53]:
predict=model.predict(input_test)

In [54]:
lossy2=r2_score(target_test,predict)

In [55]:
lossy2

0.8473063525029869

In [58]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)]
# Number of features to consider at every split
max_features = ['sqrt','log2']
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(1, 100,2)]
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10,14]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4,6,8]
# Create the random grid
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
              'criterion':['squared_error', 'friedman_mse', 'absolute_error', 'poisson']}
print(random_grid)

{'n_estimators': [200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000], 'max_features': ['sqrt', 'log2'], 'max_depth': [1, 100], 'min_samples_split': [2, 5, 10, 14], 'min_samples_leaf': [1, 2, 4, 6, 8], 'criterion': ['squared_error', 'friedman_mse', 'absolute_error', 'poisson']}


In [57]:
rf=RandomForestRegressor()
rf_randomcv=RandomizedSearchCV(estimator=rf,param_distributions=random_grid,n_iter=100,cv=3,verbose=2,
                               random_state=100,n_jobs=-1)
### fit the randomized model
rf_randomcv.fit(inputs_train,target_train)

Fitting 3 folds for each of 100 candidates, totalling 300 fits


KeyboardInterrupt: 

In [ ]:
rf_randomcv.best_params_

In [ ]:
best_random_grid=rf_randomcv.best_estimator_